# 🧵 Strands Agents SDK Tutorial (Powered by Groq & `openai/gpt-oss-120b`)

Welcome to this step-by-step hands-on guide for building AI agents using **Strands Agents** and **Groq Cloud**. 

Get keys from https://console.groq.com/


We will use OpenAI's open-weight model **`openai/gpt-oss-120b`** served via Groq's LPU hardware, providing ultra-fast inference speed and robust reasoning capabilities for multi-step tool calls.

### Setup Prerequisites

Install the required Python packages:

```bash
pip install strands-agents tavily-python openai requests python-dotenv

In [1]:
import os
from dotenv import load_dotenv

# Strands Core Imports
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from pydantic import BaseModel, Field
from tavily import TavilyClient

# Load environment variables (.env)
load_dotenv()

# Initialize Gemini model provider inside Strands
groq_model = OpenAIModel(
    model_id="openai/gpt-oss-120b",
    client_args={
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1"
    }
)

In [2]:
# !pip install gradio

## Level 1: Basic Text Execution

**Concept:** In Strands, the `Agent` class handles the agent loop. Calling an agent instance directly like a function (`agent("prompt")`) sends the user input to the model.

In [3]:
# Level 1 Example: Basic Agent Call
agent = Agent(model=groq_model)

response = agent("Explain what an API is in 2 concise sentences.")

print("--- Level 1 Response ---")
print(response)

We need to answer concisely: two sentences explaining API. Provide clear definition.An API (Application Programming Interface) is a set of defined rules and protocols that allows one software component to communicate with and request services from another. It abstracts the underlying implementation so developers can interact with functionality (like data retrieval, processing, or external services) through simple, standardized calls.--- Level 1 Response ---
An API (Application Programming Interface) is a set of defined rules and protocols that allows one software component to communicate with and request services from another. It abstracts the underlying implementation so developers can interact with functionality (like data retrieval, processing, or external services) through simple, standardized calls.



## Level 2: Setting System Prompts

**Concept:** Guide the model's persona, tone, or operational rules using the `system_prompt` parameter on the `Agent`.

In [4]:
# Level 2 Example: Custom System Prompt
tutor_agent = Agent(
    model=groq_model,
    system_prompt="You are a strict, highly accurate math tutor. Keep answers brief and clear."
)

response = tutor_agent("Is 107 a prime number? Show brief logic.")

print("--- Level 2 Response ---")
print(response)

The user asks: Is 107 a prime number? Show brief logic. We need to answer: Yes, 107 is prime. Provide brief reasoning: check divisibility up to sqrt(107) ≈10.34, test primes 2,3,5,7. Not divisible. So it's prime. Keep answer brief and clear.Yes, 107 is prime.

**Logic:**  
- √107 ≈ 10.3, so test only primes ≤ 10: 2, 3, 5, 7.  
- 107 is odd → not divisible by 2.  
- Sum of digits = 1+0+7 = 8 → not divisible by 3.  
- Does not end in 0 or 5 → not divisible by 5.  
- 107 ÷ 7 ≈ 15.29, not an integer.  

No divisors found; therefore 107 has no factors other than 1 and itself.--- Level 2 Response ---
Yes, 107 is prime.

**Logic:**  
- √107 ≈ 10.3, so test only primes ≤ 10: 2, 3, 5, 7.  
- 107 is odd → not divisible by 2.  
- Sum of digits = 1+0+7 = 8 → not divisible by 3.  
- Does not end in 0 or 5 → not divisible by 5.  
- 107 ÷ 7 ≈ 15.29, not an integer.  

No divisors found; therefore 107 has no factors other than 1 and itself.



## Level 3: Enforcing Structured Output

**Concept:** Strands supports type-safe structured output using Pydantic schemas. Pass the Pydantic model class to `structured_output_model` when instantiating the agent.

In [5]:
# 1. Define target output structure using Pydantic
class Flashcard(BaseModel):
    term: str = Field(description="The term or concept name")
    definition: str = Field(description="A clear 1-sentence definition")
    example: str = Field(description="A real-world example")

# 2. Attach schema to Strands Agent
structured_agent = Agent(
    model=groq_model,
    structured_output_model=Flashcard
)

response = structured_agent("Generate a flashcard for the concept of 'Recursion'.")

print("--- Level 3 Output (Structured Pydantic Model) ---")
print(response)

We need to generate flashcard using the structured output tool. Must include definition, example, term. Use the function Flashcard. We'll call the tool.
Tool #1: Flashcard
--- Level 3 Output (Structured Pydantic Model) ---
{"term":"Recursion","definition":"Recursion is a programming technique where a function calls itself directly or indirectly to solve a problem by breaking it into smaller, similar subproblems.","example":"A classic example is computing the factorial of n: factorial(n) = n * factorial(n-1) with the base case factorial(0) = 1."}


## Level 4: Conversational Memory / Multi-Turn Chat

**Concept:** Strands `Agent` instances automatically maintain conversation history across calls.

In [6]:
chat_agent = Agent(model=groq_model)

# Turn 1
reply_1 = chat_agent("My favorite programming language is Python.")
print("Agent:", reply_1)

# Turn 2 (Agent remembers state from Turn 1)
reply_2 = chat_agent("What is my favorite language?")
print("\nAgent:", reply_2)

The user says "My favorite programming language is Python." No question. Likely a chit-chat. We can respond positively, ask about why they like Python, discuss features. Keep friendly.That’s a great choice! Python’s readability, massive ecosystem, and versatility make it a favorite for many developers—from web development and data science to automation and AI. 

Do you have a particular project or domain where you enjoy using Python the most? Or maybe a favorite library or framework that

reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


 you keep coming back to? 🚀Agent: That’s a great choice! Python’s readability, massive ecosystem, and versatility make it a favorite for many developers—from web development and data science to automation and AI. 

Do you have a particular project or domain where you enjoy using Python the most? Or maybe a favorite library or framework that you keep coming back to? 🚀

The user asks: "What is my favorite language?" They previously said "My favorite programming language is Python." So answer: Python. Should be concise.Your favorite programming language is **Python**.
Agent: Your favorite programming language is **Python**.



## Level 5: Basic Tool Calling with `@tool` (Simple Example)

**Concept:** Custom tools in Strands are standard Python functions decorated with `@tool`. The SDK reads docstrings and type hints to inform `openai/gpt-oss-120b` when and how to invoke them.

In this simple example, we give the model a tool to convert temperatures between Celsius and Fahrenheit.

In [7]:
# -------------------------------------------------------------------------
# Step 1: Define a simple custom tool using @tool
# -------------------------------------------------------------------------

@tool
def convert_celsius_to_fahrenheit(celsius: float) -> dict:
    """Converts a temperature from Celsius to Fahrenheit.
    
    Args:
        celsius: The temperature in degrees Celsius (e.g., 25.0).
    """
    fahrenheit = (celsius * 9 / 5) + 32
    print(f"\n[TOOL EXECUTED] Converted {celsius}°C -> {fahrenheit}°F")
    return {
        "celsius": celsius,
        "fahrenheit": fahrenheit
    }

# -------------------------------------------------------------------------
# Step 2: Attach the tool to a Strands Agent
# -------------------------------------------------------------------------

converter_agent = Agent(
    model=groq_model,
    tools=[convert_celsius_to_fahrenheit],
    system_prompt="You are a helpful conversion assistant. Use tools to perform temperature calculations."
)

# -------------------------------------------------------------------------
# Step 3: Prompt the agent
# -------------------------------------------------------------------------

user_prompt = "What is 37 degrees Celsius in Fahrenheit?"
print(f"User Request: {user_prompt}")

response = converter_agent(user_prompt)

print("\n--- Level 5 Simple Response ---")
print(response)

User Request: What is 37 degrees Celsius in Fahrenheit?


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


The user asks conversion: 37°C to Fahrenheit. Use tool.
Tool #1: convert_celsius_to_fahrenheit

[TOOL EXECUTED] Converted 37.0°C -> 98.6°F
We have the conversion result. Need to respond with answer.37 °C is equal to **98.6 °F**.
--- Level 5 Simple Response ---
37 °C is equal to **98.6 °F**.



# MultiStep Tool Call
In Strands, a 2-step tool call occurs when a model uses the result of an initial tool function as the argument for a second tool.

To make tool execution sequential instead of running concurrently, pass ```SequentialToolExecutor()``` into the Agent's tool_executor argument.

In [8]:
from strands.tools.executors import SequentialToolExecutor

# -------------------------------------------------------------------------
# Step 1: Define Interdependent Tools
# -------------------------------------------------------------------------

@tool
def get_user_favorite_city(user_id: str) -> dict:
    """Retrieves the user's favorite city from a database given their User ID.
    
    Args:
        user_id: The ID of the user (e.g., 'usr_101').
    """
    print(f"\n[STEP 1 TOOL] Querying database for user: '{user_id}'...")
    
    # Mock Database lookup
    user_db = {
        "usr_101": "Tokyo",
        "usr_102": "Paris",
        "usr_103": "Kathmandu"
    }
    
    city = user_db.get(user_id, "Unknown")
    return {"user_id": user_id, "favorite_city": city}


@tool
def get_city_attractions(city_name: str) -> dict:
    """Fetches popular tourist attractions for a given city name.
    
    Args:
        city_name: The target city name (e.g., 'Tokyo', 'Paris').
    """
    print(f"\n[STEP 2 TOOL] Fetching top attractions for city: '{city_name}'...")
    
    attractions_db = {
        "Tokyo": ["Senso-ji Temple", "Shibuya Crossing", "Tokyo Tower"],
        "Paris": ["Eiffel Tower", "Louvre Museum", "Notre-Dame"],
        "Kathmandu": ["Boudhanath Stupa", "Pashupatinath Temple", "Swayambhunath"]
    }
    
    places = attractions_db.get(city_name, ["No places found."])
    return {"city": city_name, "top_attractions": places}


# -------------------------------------------------------------------------
# Step 2: Configure Agent with Sequential Tool Executor
# -------------------------------------------------------------------------

agent = Agent(
    model=groq_model,
    tools=[get_user_favorite_city, get_city_attractions],
    tool_executor=SequentialToolExecutor(),  # Enforces sequential step-by-step tool execution
    system_prompt=(
        "You are a travel assistant. Execute tools sequentially when information depends on a prior step."
    )
)

# -------------------------------------------------------------------------
# Step 3: Run the 2-Step Sequential Query
# -------------------------------------------------------------------------

if __name__ == "__main__":
    # Query requires: Step 1 (find city for usr_101) -> Step 2 (get attractions for that city)
    user_query = "What are the top tourist attractions in the favorite city of user usr_101?"
    print(f"User Request: {user_query}")

    response = agent(user_query)

    print("\n--- Final Agent Response ---")
    print(response)

User Request: What are the top tourist attractions in the favorite city of user usr_101?


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


We need to get user's favorite city then get attractions. Use functions sequentially. First call get_user_favorite_city with user_id 'usr_101'.
Tool #1: get_user_favorite_city

[STEP 1 TOOL] Querying database for user: 'usr_101'...


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


We have user's favorite city: Tokyo. Now need to fetch popular tourist attractions for Tokyo. Use get_city_attractions.
Tool #2: get_city_attractions

[STEP 2 TOOL] Fetching top attractions for city: 'Tokyo'...
Here are the top tourist attractions in **Tokyo**, the favorite city of user **usr_101**:

1. **Senso‑ji Temple** – A historic Buddhist temple in Asakusa, famous for its vibrant gate and bustling market street.  
2. **Shibuya Crossing** – The iconic “scramble” intersection known for its massive crowds and neon lights.  
3. **Tokyo Tower** – A landmark communications tower offering panoramic city views from its observation decks.
--- Final Agent Response ---
Here are the top tourist attractions in **Tokyo**, the favorite city of user **usr_101**:

1. **Senso‑ji Temple** – A historic Buddhist temple in Asakusa, famous for its vibrant gate and bustling market street.  
2. **Shibuya Crossing** – The iconic “scramble” intersection known for its massive crowds and neon lights.  
3. **

# Assignment 2: Strands Travel Planner

Build an AI agent using **Strands Agents** that creates a one-day travel plan for a city.

The agent must:
1. Use a **weather tool** to get the current weather.
2. Use a **web/search tool** to find **3 popular attractions**.
3. Use a **calculator tool** to estimate the total cost of visiting the attractions.
4. Combine the results into a concise **one-day itinerary**.

#### Grading Rubric — 5 Marks

| Criteria | Marks |
|---|---:|
| Strands agent setup and configuration | 1 |
| Correct use of weather and search tools | 1 |
| Correct use of calculator/tool calling | 1 |
| Accurate itinerary and cost calculation | 1 |
| Code quality and clear final output | 1 |
| **Total** | **5** |



# Assignment 3: Strands Travel Planner Chatbot

Build an interactive **Gradio chatbot** powered by **Strands Agents** that helps users plan a one-day trip to any city.

The chatbot must:
1. Accept a **city name** from the user.
2. Use a **weather tool** to retrieve the city's current weather.
3. Use a **search tool** to find **3 popular attractions**.
4. Use a **calculator/tool** to estimate the total visit cost.
5. Present the results as a clear **one-day itinerary** through the Gradio chat interface.

### Grading Rubric — 5 Marks

| Criteria | Marks |
|---|---:|
| Strands agent implementation | 1 |
| Gradio chatbot UI | 1 |
| Correct use of tools | 1 |
| Accurate itinerary and cost calculation | 1 |
| Code quality and user-friendly responses | 1 |
| **Total** | **5** |

## TAVILY example (token intensive)

**Concept:** Custom tools in Strands are standard Python functions decorated with `@tool`. The SDK reads docstrings and type hints to inform Gemini when and how to invoke them.

In [33]:
# Initialize Tavily
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# Define custom tool with Strands @tool decorator
@tool
def search_web(query: str) -> dict:
    """Performs a web search using Tavily to get real-time information.
    
    Args:
        query: The search query string.
    """
    print(f"\n[TOOL EXECUTED] Searching web for: '{query}'...")
    response = tavily_client.search(query=query, search_depth="basic", max_results=2)
    
    results = [
        {"title": r.get("title"), "content": r.get("content"), "url": r.get("url")}
        for r in response.get("results", [])
    ]
    return {"results": results}

# Pass the tool directly to the agent
search_agent = Agent(
    model=groq_model,
    tools=[search_web],
    system_prompt="You are a news researcher. Use search_web to find live facts and cite sources."
)

response = search_agent("What are the latest key updates on AI agent frameworks?")

print("\n--- Level 5 Agent Response ---")
print(response)

We need latest key updates on AI agent frameworks. Must use web search to get recent info. We'll search.
Tool #1: search_web

[TOOL EXECUTED] Searching web for: 'latest updates AI agent frameworks 2024'...


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


We need latest key updates on AI agent frameworks. Use sources. The search gave two articles: one about AI agent frameworks outperforming categories (crypto context) maybe not relevant. Another about best open source frameworks in 2026, includes LangGraph, CrewAI, Mastra. Need latest updates: maybe other recent news like LangChain releases, AutoGPT, OpenAI's Assistants API, etc. Let's search more.
Tool #2: search_web

[TOOL EXECUTED] Searching web for: '2024 AI agent framework release LangGraph 2024 update'...


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


We need latest key updates on AI agent frameworks. Need to compile recent developments: LangGraph, CrewAI, Mastra, GraphBit, LangChain updates, maybe AutoGPT, BabyAGI, ReAct, etc. Use sources. Let's search more.
Tool #3: search_web

[TOOL EXECUTED] Searching web for: '2024 release AI agent framework CrewAI 2024 update'...


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


We need latest key updates on AI agent frameworks. We have sources: article about best open source frameworks 2026 (but includes 2024-2026 updates). Also LangGraph updates. Also CrewAI press release. Need to summarize key updates: new frameworks like LangGraph, CrewAI enterprise platform, GraphBit (Rust core), Mastra TypeScript, etc. Provide citations.

Let's open the open-source frameworks article for more detail.

exception=<APIError> | event loop cycle failed


EventLoopException: Tool call validation failed: tool call validation failed: parameters for tool search_web did not match schema: errors: [missing properties: 'query']